<a href="https://colab.research.google.com/github/aboettcher-sig/morocco-forests-sig/blob/main/scripts/utility/pull_planet_weekly_cropped.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pull Planet weekly basemaps clipped to an AOI, into Drive

Uses the **Orders API** so the clip happens on Planet's side: you send the AOI
geometry, Planet returns one raster per week covering only that area. No whole
quad tiles are downloaded.

Flow: authenticate -> load AOI -> find the weekly series and its mosaics ->
for each week, submit a basemaps order (`merge` + `clip` to the AOI), wait for
it to finish, download the clipped result into a Drive folder.

Prerequisites:
- A Colab secret named `morroco_planet` holding the Planet API key.
- The AOI as a `.gpkg` in Drive.
- A target Drive output folder.

Edit only the CONFIG cell.

In [1]:
!pip -q install geopandas pyogrio shapely

In [2]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import geopandas as gpd
import requests
from shapely.geometry import mapping
from shapely.ops import unary_union

## CONFIG

The only cell to edit. Leave `START`/`END` as `None` to order every week in
the series; set them (YYYY-MM-DD, START inclusive, END exclusive) to limit the
range. Each week is one order, so a wide range means many orders.

In [6]:
SECRET_NAME = "morroco_planet"  # Colab secret holding the Planet API key (spelling matches the secret)

AOI_GPKG = "/content/drive/MyDrive/morocco_forests/test_region.gpkg"  # AOI polygon in Drive
AOI_LAYER = None                                              # gpkg layer name, or None for the first layer

OUT_DIR = "/content/drive/MyDrive/morocco_forests/planet_weekly_cropped"  # where clipped rasters are written

SERIES_NAME_FILTER = "weekly"   # case-insensitive substring to locate the weekly series
START = None                    # e.g. "2024-06-01", or None for all
END = None                      # e.g. "2024-09-01", or None for all

POLL_SECONDS = 10               # wait between order status checks

## Authenticate

Planet uses HTTP basic auth with the API key as the username and an empty
password. One session carries the auth; `_get`/`_post` back off on the 429
rate-limit response and surface the response body on other errors.

In [4]:
from google.colab import userdata

try:
    API_KEY = userdata.get(SECRET_NAME)
except Exception as e:
    raise RuntimeError(
        f"Could not read Colab secret '{SECRET_NAME}'. Add it via the key icon "
        f"in the left sidebar and enable notebook access."
    ) from e

BASE = "https://api.planet.com/basemaps/v1"
ORDERS = "https://api.planet.com/compute/ops/orders/v2"

session = requests.Session()
session.auth = (API_KEY, "")


def _get(url, params=None, stream=False):
    for attempt in range(6):
        r = session.get(url, params=params, stream=stream, timeout=180)
        if r.status_code == 429:
            time.sleep(float(r.headers.get("Retry-After", 2 ** attempt)))
            continue
        if not r.ok:
            raise RuntimeError(f"GET {url} -> {r.status_code} {r.text[:500]}")
        return r
    raise RuntimeError(f"GET {url} -> {r.status_code} {r.text[:500]}")


def _post(url, payload):
    for attempt in range(6):
        r = session.post(url, json=payload, timeout=180)
        if r.status_code == 429:
            time.sleep(float(r.headers.get("Retry-After", 2 ** attempt)))
            continue
        if not r.ok:
            raise RuntimeError(f"POST {url} -> {r.status_code} {r.text[:500]}")
        return r
    raise RuntimeError(f"POST {url} -> {r.status_code} {r.text[:500]}")


def paginate(url, item_key, params=None):
    while url:
        payload = _get(url, params=params).json()
        params = None
        for item in payload.get(item_key, []):
            yield item
        url = payload.get("_links", {}).get("_next")

## Mount Drive and load the AOI

The AOI is dissolved to a single geometry in EPSG:4326 and converted to GeoJSON
for the order's clip. The clip tool rejects polygons with holes, multiple
exterior rings, or more than 1500 vertices.

In [7]:
from google.colab import drive

drive.mount("/content/drive")

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

aoi = gpd.read_file(AOI_GPKG, layer=AOI_LAYER) if AOI_LAYER else gpd.read_file(AOI_GPKG)
aoi = aoi.to_crs(4326)
aoi_geom = unary_union(aoi.geometry.values)
aoi_geojson = mapping(aoi_geom)
print(f"AOI features: {len(aoi)}  bounds: {tuple(round(v, 5) for v in aoi.total_bounds)}")
print(f"geometry type: {aoi_geojson['type']}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
AOI features: 1  bounds: (np.float64(-5.4759), np.float64(35.39974), np.float64(-5.46506), np.float64(35.40673))
geometry type: Polygon


## Find the weekly series and select mosaics

Lists series, filters by name, and takes the mosaics in the chosen series. If
`START`/`END` are set, keeps the weeks whose first-acquired date is in range;
if that leaves nothing, prints the series' actual coverage instead of ordering.

In [9]:
all_series = list(paginate(f"{BASE}/series/", "series"))
matches = [s for s in all_series if SERIES_NAME_FILTER.lower() in s.get("name", "").lower()]
if not matches:
    raise RuntimeError(
        f"No series matched '{SERIES_NAME_FILTER}'. Available: "
        + ", ".join(sorted(s.get("name", "") for s in all_series))
    )
series = matches[0]
print(f"series: {series['name']} ({series['id']})")

links = series.get("_links", {})
mosaics_url = links.get("mosaics") or next(
    (v for k, v in links.items() if isinstance(v, str) and "/mosaics" in v and k != "_self"),
    f"{BASE}/series/{series['id']}/mosaics/",
)
mosaics = list(paginate(mosaics_url, "mosaics"))


def _parse(ts):
    return datetime.fromisoformat(ts.replace("Z", "+00:00"))


ordered = sorted(mosaics, key=lambda m: m["first_acquired"])
if START or END:
    lo = datetime.fromisoformat(START).replace(tzinfo=timezone.utc) if START else datetime.min.replace(tzinfo=timezone.utc)
    hi = datetime.fromisoformat(END).replace(tzinfo=timezone.utc) if END else datetime.max.replace(tzinfo=timezone.utc)
    selected = [m for m in ordered if lo <= _parse(m["first_acquired"]) < hi]
else:
    selected = ordered

if not selected:
    raise SystemExit(
        f"No mosaics in {START}..{END}. Series covers "
        f"{ordered[0]['first_acquired'][:10]} to {ordered[-1]['first_acquired'][:10]}. "
        f"Adjust START/END."
    )

print(f"{len(mosaics)} mosaics in series; ordering {len(selected)} weeks "
      f"({selected[0]['first_acquired'][:10]} to {selected[-1]['first_acquired'][:10]})")

series: PS sen2_normalized_analytic_8b_sr weekly subscription (800f630f-9378-45dd-9028-dba68b70d841)
108 mosaics in series; ordering 108 weeks (2024-07-29 to 2026-08-17)


## Order, wait, and download the clipped weeks

For each week: submit a basemaps order that merges the intersecting quads and
clips to the AOI, poll until it finishes, then download the result files into
`OUT_DIR/<mosaic_name>/`. The clipped raster covers only the AOI.

In [ ]:
END_STATES = {"success", "partial", "failed", "cancelled"}


def order_payload(mosaic_name):
    return {
        "name": f"weekly_clip_{mosaic_name}",
        "source_type": "basemaps",
        "order_type": "partial",
        "products": [{"mosaic_name": mosaic_name, "geometry": aoi_geojson}],
        "tools": [{"merge": {}}, {"clip": {}}],
    }


def poll_order(order_id):
    url = f"{ORDERS}/{order_id}"
    while True:
        state = _get(url).json()["state"]
        if state in END_STATES:
            return _get(url).json()
        time.sleep(POLL_SECONDS)


def download_results(order_json, dest_dir):
    dest_dir.mkdir(parents=True, exist_ok=True)
    saved = []
    for r in order_json.get("_links", {}).get("results", []):
        loc = r.get("location")
        if not loc:
            continue
        name = (r.get("name") or loc.split("?")[0]).split("/")[-1]
        dest = dest_dir / name
        with _get(loc, stream=True) as resp, open(dest, "wb") as f:
            for chunk in resp.iter_content(chunk_size=1 << 20):
                if chunk:
                    f.write(chunk)
        saved.append(dest)
    return saved


made = []
for m in selected:
    name = m["name"]
    print(f"ordering {name} ...", flush=True)
    order_id = _post(ORDERS, order_payload(name)).json()["id"]
    final = poll_order(order_id)
    if final["state"] in {"failed", "cancelled"}:
        print(f"  {final['state']}: {name} (order {order_id})")
        continue
    files = download_results(final, Path(OUT_DIR) / name)
    made += files
    print(f"  {final['state']}: {len(files)} file(s) -> {OUT_DIR}/{name}")

print(f"\nDone. {len(made)} files across {len(selected)} weeks -> {OUT_DIR}")

ordering ps_weekly_sen2_normalized_analytic_8b_sr_subscription_2024-07-29_2024-08-05_mosaic ...
  success: 6 file(s) -> /content/drive/MyDrive/morocco_forests/planet_weekly_cropped/ps_weekly_sen2_normalized_analytic_8b_sr_subscription_2024-07-29_2024-08-05_mosaic
ordering ps_weekly_sen2_normalized_analytic_8b_sr_subscription_2024-08-05_2024-08-12_mosaic ...
  success: 6 file(s) -> /content/drive/MyDrive/morocco_forests/planet_weekly_cropped/ps_weekly_sen2_normalized_analytic_8b_sr_subscription_2024-08-05_2024-08-12_mosaic
ordering ps_weekly_sen2_normalized_analytic_8b_sr_subscription_2024-08-12_2024-08-19_mosaic ...
  success: 6 file(s) -> /content/drive/MyDrive/morocco_forests/planet_weekly_cropped/ps_weekly_sen2_normalized_analytic_8b_sr_subscription_2024-08-12_2024-08-19_mosaic
ordering ps_weekly_sen2_normalized_analytic_8b_sr_subscription_2024-08-19_2024-08-26_mosaic ...
  success: 6 file(s) -> /content/drive/MyDrive/morocco_forests/planet_weekly_cropped/ps_weekly_sen2_normalized_a